In [1]:
%pip install langchain-classic openai langchain-openai pandas scikit-learn catboost xgboost seaborn matplotlib joblib sentence-transformers faiss-cpu torch langchain-community tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from catboost import CatBoostRegressor, Pool
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from langchain_core.tools import Tool  # Changed import path
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.agents.agent import AgentExecutor
from langchain.agents import create_agent
from langchain_classic import hub
from langchain_core.documents import Document
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import matplotlib.pyplot as plt
import pickle
import faiss
import torch
from openai import OpenAI
from tqdm import tqdm
import warnings
from dotenv import load_dotenv
import os
RANDOM_STATE = 42
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

In [4]:
# Load environment variables from .env file (if uploaded to Colab session)
load_dotenv()
print("Environment variables from .env file loaded.")

try:
    # Initialize OpenAI Client (will use OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI Client initialized successfully.")
except Exception as e:
    print("ERROR: Failed to initialize OpenAI client. Ensure 'OPENAI_API_KEY' environment variable is set.")
    raise e

Environment variables from .env file loaded.
OpenAI Client initialized successfully.


In [5]:
df = pd.read_csv('car_prices_extended_eda.csv')
# The 15 features you want to use for the final model
FINAL_FEATURES = [
    'year', 'make', 'model', 'body', 'transmission', 'state',
    'odometer', 'color', 'interior', 'mmr', 'car_age',
    'sale_year', 'sale_month', 'condition_category', 'is_weekend'
]
TARGET_COLUMN = 'sellingprice'
ALL_COLS_TO_KEEP = FINAL_FEATURES + [TARGET_COLUMN]

In [6]:
# --- 1. Filter the DataFrame to keep only the 15 features + target ---
# This drops columns like 'trim', 'seller', 'mileage_per_year', etc.
df = df[ALL_COLS_TO_KEEP]

In [7]:
# --- 2. Log Transformation (CRITICAL: Must be applied before training) ---
# As seen in your notebook, odometer and car_age are log-transformed.
# MMR is usually highly correlated and can benefit from log transformation too,
# but we will stick to your observed log transformations for odometer and car_age.
skewed_cols = ["odometer", "car_age"]
df[skewed_cols] = np.log1p(df[skewed_cols])

In [8]:
# --- 3. Define X and y ---
# Transform the target to log-scale for training
y = np.log1p(df[TARGET_COLUMN])
X = df.drop(columns=[TARGET_COLUMN])

In [9]:
# --- 4. Verify Features and Save Model ---
print("Final X Features:", X.columns.tolist())
# Expected: ['year', 'make', 'model', 'body', 'transmission', 'state', 'odometer', 'color', 'interior', 'mmr', 'car_age', 'sale_year', 'sale_month', 'condition_category', 'is_weekend']

Final X Features: ['year', 'make', 'model', 'body', 'transmission', 'state', 'odometer', 'color', 'interior', 'mmr', 'car_age', 'sale_year', 'sale_month', 'condition_category', 'is_weekend']


In [10]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
# Train with Pool (efficient)
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
# Add other columns that should be treated as categorical even if their dtype is not 'object'
# e.g., 'year', 'sale_year', 'sale_month', 'is_weekend' are often treated as categorical
additional_cat_features = ['year', 'sale_year', 'sale_month', 'is_weekend']
for col in additional_cat_features:
    if col not in cat_features and col in X_train.columns:
        cat_features.append(col)

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features) # Create test pool for eval_set

model = CatBoostRegressor(
    iterations=700,
    depth=8,
    learning_rate=0.2,
    random_seed=42,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100
)
model.fit(train_pool, eval_set=test_pool) # Pass eval_set to fit method

0:	learn: 0.7314291	test: 0.7285912	best: 0.7285912 (0)	total: 224ms	remaining: 2m 36s
100:	learn: 0.2050998	test: 0.2076518	best: 0.2076518 (100)	total: 17.1s	remaining: 1m 41s
200:	learn: 0.1985641	test: 0.2052187	best: 0.2052187 (200)	total: 34.9s	remaining: 1m 26s
300:	learn: 0.1946372	test: 0.2044403	best: 0.2044253 (299)	total: 52.6s	remaining: 1m 9s
400:	learn: 0.1909821	test: 0.2040710	best: 0.2040665 (386)	total: 1m 11s	remaining: 53.1s
500:	learn: 0.1878229	test: 0.2038431	best: 0.2038431 (500)	total: 1m 29s	remaining: 35.6s
600:	learn: 0.1850706	test: 0.2036838	best: 0.2036763 (594)	total: 1m 47s	remaining: 17.8s
699:	learn: 0.1827231	test: 0.2035891	best: 0.2035693 (689)	total: 2m 6s	remaining: 0us

bestTest = 0.2035693295
bestIteration = 689

Shrink model to first 690 iterations.


In [25]:
y_train_pred_log = model.predict(X_train)
y_test_pred_log = model.predict(X_test)

# Actuals (already log-transformed in Cell 20)
y_train_actual_log = y.loc[X_train.index]
y_test_actual_log = y.loc[X_test.index]

# 1. Back-Transform predictions and actuals to dollar scale
y_train_actual_dollars = np.expm1(y_train_actual_log)
y_test_actual_dollars = np.expm1(y_test_actual_log)
y_train_pred_dollars = np.expm1(y_train_pred_log)
y_test_pred_dollars = np.expm1(y_test_pred_log)

# 2. Calculate metrics (MAE, RMSE) on the DOLLAR SCALE
train_mae = mean_absolute_error(y_train_actual_dollars, y_train_pred_dollars)
test_mae = mean_absolute_error(y_test_actual_dollars, y_test_pred_dollars)
train_rmse = np.sqrt(mean_squared_error(y_train_actual_dollars, y_train_pred_dollars))
test_rmse = np.sqrt(mean_squared_error(y_test_actual_dollars, y_test_pred_dollars))

print(f"Train MAE: ${train_mae:.0f} | Test MAE: ${test_mae:.0f} | Gap: {((train_mae - test_mae) / test_mae * 100):+.1f}%")
print(f"Train RMSE: ${train_rmse:.0f} | Test RMSE: ${test_rmse:.0f} | Gap: {((train_rmse - test_rmse) / test_rmse * 100):+.1f}%")

# Threshold: If test/train ratio <0.85 (test 15% worse), overfitting likely
if test_rmse / train_rmse < 0.85:
    print("⚠️ Potential Overfitting: Test underperforms train significantly.")
else:
    print("✅ Good Fit: Metrics are comparable.")

Train MAE: $984 | Test MAE: $1005 | Gap: -2.1%
Train RMSE: $1814 | Test RMSE: $1822 | Gap: -0.4%
✅ Good Fit: Metrics are comparable.


In [ ]:
# --- Auto-Calibration for Low-Balling ---
# Calculate the systematic bias in log space
log_residuals = y_train_actual_log - y_train_pred_log
bias_correction = np.mean(log_residuals)

print(f"\n--- Bias Correction Factor ---")
print(f"Calculated Bias: {bias_correction:.5f}")
print(f"Applying this bias to the model directly... (predictions will be shifted automatically)")

# Bake the bias into the model!
model.set_scale_and_bias(1.0, bias_correction)

# Verify Improvement
# Now model.predict() will return (log_raw + bias)
y_test_pred_baked_log = model.predict(X_test)
y_test_pred_baked_dollars = np.expm1(y_test_pred_baked_log)

test_mae_corrected = mean_absolute_error(y_test_actual_dollars, y_test_pred_baked_dollars)

print(f"\nOriginal Test MAE: ${test_mae:.0f}")
print(f"Corrected Test MAE: ${test_mae_corrected:.0f} (Improvement: {test_mae - test_mae_corrected:.0f} dollars)")

In [26]:
# Save the model
MODEL_SAVE_PATH = 'catboost_simplified.cbm'
model.save_model(MODEL_SAVE_PATH)
print(f"Successfully saved the 15-feature model to: {MODEL_SAVE_PATH}")

Successfully saved the 15-feature model to: catboost_simplified.cbm


In [27]:
# --- Configuration ---
NEWS_FILE_PATH = "News_dataset.csv"
FAISS_INDEX_PATH = "faiss_news_index" # Folder name for the saved index
TITLE_COLUMN = 'title'
CONTENT_COLUMN = 'content'

# --- Index Creation Function ---
def build_and_save_index():
    print(f"Starting index creation from {NEWS_FILE_PATH}...")
    try:
        df = pd.read_csv(NEWS_FILE_PATH)

        # --- CRITICAL FIX: Handle Missing Values (NaN) ---
        # 1. Fill NaN values in the text columns with an empty string
        df['headline'] = df['headline'].fillna('')
        df['short_description'] = df['short_description'].fillna('')

        # 2. Combine textual columns into a single 'content' column
        # Use str() conversion for robustness, though fillna should cover it.
        df['content'] = df['headline'].astype(str) + ". " + df['short_description'].astype(str)

        # 3. Prepare LangChain Documents
        documents = []
        for index, row in df.iterrows():
            # Use only rows where content is not an empty string after cleaning
            if row['content'].strip() == '.': # If both were empty, content might just be ". "
                continue

            # Metadata includes all non-content columns
            metadata = {
                "headline": row['headline'],
                "category": row['category'],
                "year": row['year'],
                "month": row['month'],
                "day": row['day']
            }
            documents.append(
                Document(page_content=row['content'], metadata=metadata)
            )

        # 4. Split documents
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        docs = text_splitter.split_documents(documents)

        # 5. Create OpenAI embeddings
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        # 6. Create and save FAISS index
        vector_store = FAISS.from_documents(docs, embeddings)
        vector_store.save_local(FAISS_INDEX_PATH)

        print(f"\n--- SUCCESS! ---")
        print(f"Processed {len(documents)} documents. FAISS index saved to: '{FAISS_INDEX_PATH}'")

    except FileNotFoundError:
        print(f"Error: The file '{NEWS_FILE_PATH}' was not found.")
    except Exception as e:
        print(f"An error occurred during indexing: {e}")

In [28]:
build_and_save_index()

Starting index creation from News_dataset.csv...


KeyboardInterrupt: 